# YorTIB-200 — Reproducibility and Publication Figures

Creates publication-ready summary tables and 300-dpi figures from the paired prediction data.

In [ ]:
import json
from pathlib import Path
import pandas as pd
import numpy as np

DATA = Path("../data/yortib_200.json")
with open(DATA, encoding="utf-8") as f:
    data = json.load(f)
if isinstance(data, dict):
    data = data.get("data", data.get("items", []))
df = pd.DataFrame(data)
print("Rows:", len(df))
df.head()


In [ ]:
baseline = read_prediction(Path("../predictions/baseline.csv"))
sovereign = read_prediction(Path("../predictions/sovereign.csv"))

df["baseline_correct"] = baseline.values == df["answer"].astype(str).str.strip().values
df["sovereign_correct"] = sovereign.values == df["answer"].astype(str).str.strip().values

out = Path("../results/figure_data")
out.mkdir(parents=True, exist_ok=True)

pd.DataFrame({
    "condition":["Baseline","Sovereignty-preserving"],
    "accuracy":[df.baseline_correct.mean(),df.sovereign_correct.mean()]
}).to_csv(out/"overall_accuracy.csv", index=False)

df[["id","focus_word","answer","baseline_correct","sovereign_correct"]].to_csv(
    out/"item_level_with_gold.csv", index=False
)
print("Figure data exported.")


In [ ]:
import matplotlib.pyplot as plt

fig_dir = Path("../figures")
fig_dir.mkdir(exist_ok=True)

conditions = ["Baseline","Sovereignty-preserving"]
acc = [df.baseline_correct.mean(), df.sovereign_correct.mean()]

plt.figure(figsize=(7,5))
bars = plt.bar(conditions, acc)
plt.ylim(0,1.08)
plt.ylabel("Accuracy")
plt.title("Overall Contextual Lexical-Semantic Accuracy")
plt.text(0, acc[0]+.025, f"{acc[0]*100:.1f}%", ha="center")
plt.text(1, acc[1]+.025, f"{acc[1]*100:.1f}%", ha="center")
plt.tight_layout()
plt.savefig(fig_dir/"Figure_1_overall_accuracy.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
focus = df.groupby("focus_word")[["baseline_correct","sovereign_correct"]].mean()

plt.figure(figsize=(8,5))
focus.plot(kind="bar", ax=plt.gca())
plt.ylim(0,1.08)
plt.ylabel("Accuracy")
plt.xlabel("Focus word")
plt.title("Accuracy by Yoruba Focus Word")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(fig_dir/"Figure_2_focus_word_accuracy.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
transitions = pd.Series({
    "Correct → Correct": int((df.baseline_correct & df.sovereign_correct).sum()),
    "Incorrect → Correct": int((~df.baseline_correct & df.sovereign_correct).sum()),
    "Correct → Incorrect": int((df.baseline_correct & ~df.sovereign_correct).sum()),
    "Incorrect → Incorrect": int((~df.baseline_correct & ~df.sovereign_correct).sum())
})

plt.figure(figsize=(8,5))
transitions.plot(kind="bar")
plt.ylabel("Number of items")
plt.xlabel("")
plt.title("Paired Item-Level Classification Transitions")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig(fig_dir/"Figure_3_paired_transitions.png", dpi=300, bbox_inches="tight")
plt.show()

print("Figures saved at 300 dpi in:", fig_dir)
